In [ ]:
This code allows to combine trajectory files into a unified dataset and clean it.

In [ ]:
import pandas as pd
import glob


files = glob.glob('../data/raw/trajectory/*.json')


df_list = [pd.read_json(f, lines=True) for f in files]


combined_df = pd.concat(df_list, ignore_index=True)

In [ ]:
import shapely.wkt as wkt

sorted_df["geometry"] = sorted_df["locationTrack"].apply(wkt.loads)

In [ ]:

import geopandas as gpd
from shapely.geometry import LineString, MultiLineString
from geopy.distance import geodesic

def process_geometry_with_timestamps(geom, timestamps, max_time_gap=300, speed_limit = 25):
    """
    Handles both LineString and MultiLineString
    CRS: EPSG:4326 (lat/lon)
    """
    
  
    if geom is None:
        return []
    if isinstance(geom, LineString):
        lines = [geom]
        
    elif isinstance(geom, MultiLineString):
        
        lines = list(geom.geoms)
    else:
        return []
    
 
    all_coords = []
    for line in lines:
        all_coords.extend(list(line.coords))
    

    if len(all_coords) != len(timestamps):
        return []
    
    segments = []
    current_segment = [all_coords[0]]
    
    for i in range(1, len(all_coords)):
        p1 = all_coords[i - 1]
        p2 = all_coords[i]
        
        t1 = pd.to_datetime(timestamps[i - 1])
        t2 = pd.to_datetime(timestamps[i])
        
        time_diff = (t2 - t1).total_seconds()
        
       
        dist = geodesic((p1[1], p1[0]), (p2[1], p2[0])).meters
        
        speed = dist / time_diff if time_diff > 0 else 0
        
    
        if time_diff > max_time_gap or speed > speed_limit:
            if len(current_segment) > 1:
                segments.append(LineString(current_segment))
            current_segment = [p2]
        else:
            current_segment.append(p2)
    
    
    if len(current_segment) > 1:
        segments.append(LineString(current_segment))
    
    return segments

In [ ]:
sorted_df["clean_segments"] = sorted_df.apply(
    lambda row: process_geometry_with_timestamps(
        row["geometry"],
        row["timestamps"]
    ),
    axis=1
)

In [ ]:
sorted_df_clean = sorted_df.explode("clean_segments").dropna(subset=["clean_segments"])

In [ ]:
gdf_clean = gpd.GeoDataFrame(
    sorted_df_clean,
    geometry="clean_segments",
    crs="EPSG:4326"
)

In [ ]:
land_gdf = gpd.read_file("../data/processed/land_data.gpkg")
land_gdf.head()

In [ ]:
gdf_clean = gdf_clean.to_crs(land_gdf.crs)

In [ ]:
land_union = gdf_land.union_all()

In [ ]:
from shapely.geometry import LineString, MultiLineString

def remove_land_parts(geom, land_geom):
    if geom is None or geom.is_empty:
        return []

    # Subtract land from trajectory
    result = geom.difference(land_geom)

    if result.is_empty:
        return []

    if isinstance(result, LineString):
        return [result]
    elif isinstance(result, MultiLineString):
        return list(result.geoms)
    else:
        return []

In [ ]:
gdf_clean["water_segments"] = gdf_clean["clean_segments"].apply(
    lambda geom: remove_land_parts(geom, land_union)
)

In [ ]:
gdf_clean_final = gdf_clean.explode("water_segments").dropna(subset=["water_segments"])


In [ ]:
gdf_clean_final = gdf_clean_final.set_geometry("water_segments")

In [ ]:
gdf_clean_final = gdf_clean_final.drop(columns=["geometry"])

In [ ]:
gdf_clean_final.to_file("../data/processed/historical_tracks.gpkg", layer="tracks", driver="GPKG")